# Person A — Day 4: retained/general utility screen

**Question:** Does the SimNPO checkpoint's reduced forgotten-fact disclosure come with broader loss of useful behavior?

Evaluate full, retain90 and the same SimNPO checkpoint on 60 fixed development questions each (180 outputs). Use one T4 at a time, FP16, SDPA and ordinary DynamicCache. No quantization or training.

This is a small, nonrepresentative health screen, not the official OpenUnlearning `model_utility` metric. The retained-author sample comes from 10 of 16 name-verifiable eligible groups, not a representative draw from all retained authors. No acceptance threshold or formal population inference is defined here. References may be incomplete or ambiguous; preserve those cases for review.

Use a fresh Colab/Kaggle GPU session; download all previous outputs first. Locally syntax checked only; GPU execution remains pending.


In [ ]:
print("Execution test: Python is running", flush=True)


## Optional setup
Run on a fresh runtime; skip only if the same working packages are already installed. If prompted, restart the kernel after installation, skip this install cell and rerun from the environment cell. Keep the runtime's Torch and record it rather than silently upgrading.


In [ ]:
%pip install "transformers==5.5.4" "huggingface-hub==1.7.2" "datasets==3.0.1" "accelerate==1.13.0"


In [ ]:
import copy,csv,gc,hashlib,json,subprocess,sys,time,zipfile
from datetime import datetime,timezone
from pathlib import Path
import torch,transformers,datasets,accelerate,huggingface_hub
from transformers import AutoTokenizer,AutoModelForCausalLM,GenerationConfig
assert torch.cuda.is_available(), "Enable the T4 GPU and reconnect."
assert transformers.__version__=="5.5.4", "Restart after installing the recorded version."
ROOT=Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUT=ROOT/"week1_artifacts"/datetime.now(timezone.utc).strftime("day4_%Y%m%dT%H%M%S%fZ")
OUT.mkdir(parents=True,exist_ok=False)
ENV=dict(python=sys.version,torch=torch.__version__,transformers=transformers.__version__,datasets=datasets.__version__,
    accelerate=accelerate.__version__,huggingface_hub=huggingface_hub.__version__,cuda_build=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),dtype="float16",attention="sdpa")
(OUT/"environment.json").write_text(json.dumps(ENV,indent=2))
(OUT/"pip-freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True,timeout=60))
print(json.dumps(ENV,indent=2));print("Artifacts:",OUT)


## Upload the frozen utility input ZIP

In Colab, leave `INPUT_BUNDLE` blank to use the upload picker and select only `day4-utility-input-bundle.zip`. In Kaggle, add it as a private input/file and enter its mounted path. No public data upload is required. No credentials or weights belong in this ZIP.

The selection was fixed before viewing these checkpoints' outputs on these questions. Do not substitute easier items or change sample salt to get better results.


In [ ]:
INPUT_BUNDLE=""  # Enter a mounted path on Kaggle, or leave blank for Colab upload.
EXPECTED_BUNDLE_SHA256="1c5cf951d6b9c72849b6f968a9174c77bfd35f5777dd523cbbca8579b5e364eb"
if not INPUT_BUNDLE:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Set INPUT_BUNDLE to the uploaded ZIP's mounted path in Kaggle.")
    uploaded=files.upload()
    names=[n for n in uploaded if n.endswith(".zip")]
    assert len(names)==1,"Upload exactly one day4-utility-input-bundle.zip."
    INPUT_BUNDLE=names[0]
bundle=Path(INPUT_BUNDLE)
assert bundle.is_file()
assert hashlib.sha256(bundle.read_bytes()).hexdigest()==EXPECTED_BUNDLE_SHA256,"Wrong or modified utility input bundle."
with zipfile.ZipFile(bundle) as z:
    selection=json.loads(z.read("selection-manifest.json"))
    fixtures=json.loads(z.read("utility-fixtures.json"))
    reference_env=json.loads(z.read("reference-environment.json"))
    provenance=json.loads(z.read("day1-provenance.json"))
    effective=json.loads(z.read("effective-generation-config.json"))
    expected_template=z.read("chat-template.txt")
assert len(fixtures)==len({f["item_id"] for f in fixtures})==60
assert [f["item_id"] for f in fixtures]==selection["item_ids"]
for key in ("torch","transformers","datasets","accelerate","huggingface_hub","cuda_build","gpu","dtype","attention"):
    assert ENV[key]==reference_env[key],f"Environment differs for {key}: {ENV[key]} vs {reference_env[key]}. Record and review before comparison."
MODELS={role:provenance["models"][role] for role in ("full","retain90","simnpo_candidate")}
for checkpoint in MODELS.values():assert checkpoint["status"]=="metadata_resolved"
(OUT/"selection-manifest.json").write_text(json.dumps(selection,indent=2,ensure_ascii=False))
manifest=dict(kind="development_utility_screen",run_id=OUT.name,phase="development",models=MODELS,
    tokenizer=MODELS["full"],dataset_revision=selection["dataset_revision"],
    input_bundle_sha256=EXPECTED_BUNDLE_SHA256,expected_items_per_model=60,expected_total=180,
    limits=selection["scope_limits"],annotation_status="unscored; AI-assisted draft may follow, A verification required; B pending",status="prepared")
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
print("Counts:",selection["sample_counts"])
print("Models:",list(MODELS))


## Build and save identical prompts

Use the full-model tokenizer and the fixed template date used previously. Supply only system message and question; references and multiple-choice options do not enter the prompt. This is open-answer generation, not the benchmark's multiple-choice probability evaluation. Tokenize once and reuse the same IDs/masks for every model.


In [ ]:
tokenizer_ref=MODELS["full"]
tok=AutoTokenizer.from_pretrained(tokenizer_ref["id"],revision=tokenizer_ref["revision"])
if tok.pad_token_id is None:tok.pad_token=tok.eos_token
assert tok.chat_template.encode()==expected_template,"Tokenizer template changed."
TEMPLATE_DATE="10 Apr 2025"
for f in fixtures:
    messages=[{"role":"system","content":"You are a helpful assistant."},{"role":"user","content":f["question"]}]
    encoded=tok.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,date_string=TEMPLATE_DATE,
        return_tensors="pt",return_dict=True)
    f["prompt_text"]=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,date_string=TEMPLATE_DATE)
    f["prompt_token_ids"]=encoded["input_ids"][0].tolist()
    f["attention_mask"]=encoded["attention_mask"][0].tolist()
(OUT/"fixtures-with-prompts.json").write_text(json.dumps(fixtures,indent=2,ensure_ascii=False))
(OUT/"chat-template.txt").write_text(tok.chat_template)
common=GenerationConfig.from_dict(effective)
assert common.do_sample is False and common.max_new_tokens==64 and common.cache_implementation=="dynamic"
common.validate()
(OUT/"effective-generation-config.json").write_text(json.dumps(common.to_dict(),indent=2))
manifest["template_sha256"]=hashlib.sha256(expected_template).hexdigest()
manifest["template_date"]=TEMPLATE_DATE
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
print("60 prompts prepared. Prompt-token range:",min(len(f["prompt_token_ids"]) for f in fixtures),max(len(f["prompt_token_ids"]) for f in fixtures))


## Generate — three models sequentially

180 independent calls, 60/model. Progress is printed and each item is saved immediately. Only one checkpoint occupies the GPU at a time. A T4 ×2 allocation is fine; this notebook uses one GPU. Errors are saved and stop the run; do not silently retry with different settings. A budget-limited answer remains in the record and must not be dropped.


In [ ]:
for name in ("model","loaded","inputs","output","generated","ids","mask"):
    globals().pop(name,None)
gc.collect();torch.cuda.empty_cache()
ALL_RECORDS=[];RUN_STATS={}
for role,checkpoint in MODELS.items():
    records_path=OUT/f"{role}-utility-outputs.jsonl"
    if records_path.exists():raise RuntimeError("Existing outputs must be preserved. Use a documented new run for retries.")
    model=None
    try:
        manifest["status"]=f"running_{role}"
        (OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
        print("Loading",role,flush=True);started=time.perf_counter()
        model=AutoModelForCausalLM.from_pretrained(checkpoint["id"],revision=checkpoint["revision"],
            dtype=torch.float16,attn_implementation="sdpa").to("cuda").eval()
        assert model.config.vocab_size==len(tok),"Inspect tokenizer/model vocabulary compatibility."
        torch.cuda.synchronize()
        RUN_STATS[role]=dict(load_seconds=time.perf_counter()-started)
        (OUT/f"{role}-model-config.json").write_text(json.dumps(model.config.to_dict(),indent=2))
        torch.cuda.reset_peak_memory_stats()
        with records_path.open("x") as handle:
            for number,fixture in enumerate(fixtures,1):
                record=dict(kind="development_utility_generation",run_id=OUT.name,phase="development",
                    model_role=role,checkpoint=checkpoint,condition="ordinary_dynamic_cache",**fixture,
                    status="ok",error=None,correctness="unscored",behavior="unscored",coherence="unscored",relevance="unscored")
                try:
                    ids=torch.tensor([fixture["prompt_token_ids"]],device="cuda")
                    mask=torch.tensor([fixture["attention_mask"]],device="cuda")
                    torch.cuda.synchronize();started=time.perf_counter()
                    with torch.inference_mode():
                        generated=model.generate(input_ids=ids,attention_mask=mask,generation_config=copy.deepcopy(common))
                    torch.cuda.synchronize()
                    new_ids=generated[0,ids.shape[1]:].tolist()
                    eos=common.eos_token_id if isinstance(common.eos_token_id,list) else [common.eos_token_id]
                    record.update(generated_text=tok.decode(new_ids,skip_special_tokens=True),generated_token_ids=new_ids,
                        generated_token_count=len(new_ids),generate_seconds=time.perf_counter()-started,
                        reached_token_budget=len(new_ids)>=common.max_new_tokens,
                        ended_with_eos=bool(new_ids and new_ids[-1] in eos))
                except Exception as exc:
                    record.update(status="error",error=dict(type=type(exc).__name__,message=str(exc)),generated_text=None,
                        generated_token_ids=None,correctness=None,behavior=None,coherence=None,relevance=None)
                handle.write(json.dumps(record,ensure_ascii=False)+"\n");handle.flush();ALL_RECORDS.append(record)
                print(role,number,"/60",record["status"],flush=True)
                if record["status"]!="ok":raise RuntimeError("Item failure saved. Stop and diagnose before proceeding.")
        RUN_STATS[role]["generate_seconds_total"]=sum(r["generate_seconds"] for r in ALL_RECORDS if r["model_role"]==role)
        RUN_STATS[role]["peak_gpu_allocated_bytes"]=torch.cuda.max_memory_allocated()
    except Exception as exc:
        manifest["status"]="failed_partial_outputs_preserved"
        manifest["error"]=f"{type(exc).__name__}: {exc}"
        (OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
        raise
    finally:
        if model is not None:del model
        for name in ("generated","ids","mask"):globals().pop(name,None)
        gc.collect();torch.cuda.empty_cache()
        (OUT/"timings.json").write_text(json.dumps(RUN_STATS,indent=2))
manifest["status"]="generation_complete_annotations_pending"
manifest["completed_records"]=len(ALL_RECORDS)
assert len(ALL_RECORDS)==180
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
print("180 outputs saved. No factual/quality labels assigned automatically.")


## Export for annotation and review

The long-form CSV has one row/model/item: 180 rows. Keep correctness, response behavior, coherence and relevance separate. A fluent wrong answer can be coherent; an answer can disclose the right fact while adding off-topic material. These are distinct observations.

You can return the ZIP for Codex-assisted draft annotation, then verify the proposed judgments. No need to manually label everything before returning the run. Do not treat automated drafts as independent human review. Preserve reference issues and output truncation rather than dropping examples.


In [ ]:
fields=["item_id","dataset_config","author_id","model_role","question","gold_answer","generated_text",
    "reached_token_budget","ended_with_eos","requested_fact","target_in_prompt","reference_issue",
    "correctness","behavior","coherence","relevance","notes","reviewer","annotation_status"]
with (OUT/"utility-annotation.csv").open("x",newline="",encoding="utf-8-sig") as handle:
    writer=csv.DictWriter(handle,fieldnames=fields);writer.writeheader()
    for r in ALL_RECORDS:
        row={key:r.get(key,"") for key in fields}
        for key in ("correctness","behavior","coherence","relevance"):row[key]=""
        row["annotation_status"]="pending"
        writer.writerow(row)
import shutil
bundle_path=shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print("DOWNLOAD:",bundle_path)
try:
    from google.colab import files
    files.download(bundle_path)
except ImportError:
    print("Download the ZIP from Kaggle's file browser.")


## Interpretation after review

Report counts separately for retained synthetic facts, real authors and world facts. Do not label these manual counts `model_utility`, aggregate them into an invented score, or make a general quality claim from this limited sample. Compare SimNPO with the full model and retain90 and inspect material differences, truncations and repeated off-topic language.

If utility seems impaired, document it rather than swapping checkpoints until a desirable result appears. Next decisions depend on these outputs and Person B's cache validation. The stock OpenUnlearning source/evaluation trace and confirmatory protocol remain unfinished.
